# Build Dataset — 28-Day Window V4

Identical pipeline to notebook 21 but aggregated over **28-day windows**.

- **Store:** CA_1 | **Depts:** FOODS_1/2/3 | **Items:** ~1,437 SKUs
- **History:** ~5 years (Jan 2011 → Apr 2016)
- **Split:** 70 / 15 / 15 | **Expected rows:** ~98K total
- **Target:** `aggregated_sales_28` — sum of daily sales over 28 days
- **Output:** `data/processed/28_Day_Dataset/`

## 1) Imports & Paths

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

WINDOW = 28
TARGET = f'aggregated_sales_{WINDOW}'

ROOT = Path('..').resolve()
M5   = ROOT / 'data' / 'raw' / 'm5'
OUT  = ROOT / 'data' / 'processed' / '28_Day_Dataset'
OUT.mkdir(parents=True, exist_ok=True)

print(f'Window : {WINDOW} days')
print(f'Target : {TARGET}')
print(f'Output : {OUT}')

Window : 28 days
Target : aggregated_sales_28
Output : C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment\data\processed\28_Day_Dataset


## 2) Load Raw Files

In [2]:
sales_df    = pd.read_csv(M5 / 'sales_train_validation.csv')
calendar_df = pd.read_csv(M5 / 'calendar.csv', parse_dates=['date'])
prices_df   = pd.read_csv(M5 / 'sell_prices.csv')

print(f'Sales shape    : {sales_df.shape}')
print(f'Calendar shape : {calendar_df.shape}')
print(f'Prices shape   : {prices_df.shape}')

Sales shape    : (30490, 1919)
Calendar shape : (1969, 14)
Prices shape   : (6841121, 4)


## 3) Filter to CA_1 / FOODS_1/2/3

In [3]:
FOOD_DEPTS = ['FOODS_1', 'FOODS_2', 'FOODS_3']
mask       = (sales_df['store_id'] == 'CA_1') & (sales_df['dept_id'].isin(FOOD_DEPTS))
sales_df   = sales_df[mask].reset_index(drop=True)
print(f'Items after filter : {len(sales_df)}')
print(sales_df['dept_id'].value_counts().to_dict())

Items after filter : 1437
{'FOODS_3': 823, 'FOODS_2': 398, 'FOODS_1': 216}


## 4) Melt Wide → Long (daily rows)

In [4]:
id_cols  = ['item_id', 'store_id', 'dept_id']
day_cols = [c for c in sales_df.columns if c.startswith('d_')]
long_df  = sales_df[id_cols + day_cols].melt(
    id_vars=id_cols, var_name='d', value_name='sales'
)
print(f'Long format shape: {long_df.shape}')

Long format shape: (2748981, 5)


## 5) Merge Calendar

In [5]:
calendar_slim = calendar_df[['d', 'date', 'wm_yr_wk', 'snap_CA',
                              'event_name_1', 'event_name_2']].copy()
long_df = long_df.merge(calendar_slim, on='d', how='left')
print(f'After calendar merge: {long_df.shape}')

After calendar merge: (2748981, 10)


## 6) Merge Sell Prices

In [6]:
prices_slim = prices_df[prices_df['store_id'] == 'CA_1'][['item_id', 'wm_yr_wk', 'sell_price']]
long_df = long_df.merge(prices_slim, on=['item_id', 'wm_yr_wk'], how='left')
long_df = long_df.sort_values(['item_id', 'date'])
long_df['sell_price'] = (
    long_df.groupby('item_id')['sell_price']
    .transform(lambda s: s.ffill().bfill())
)
long_df['sell_price'] = long_df['sell_price'].fillna(long_df['sell_price'].median())
print(f'Null sell_price after fill: {long_df["sell_price"].isna().sum()}')

Null sell_price after fill: 0


## 7) Build Event Binary Columns + is_month_end

In [7]:
EVENT_MAP = {
    'event_christmas_28':          'Christmas',
    'event_easter_28':             'Easter',
    'event_eid_al_fitr_28':        'Eid al-Fitr',
    'event_eid_al_adha_28':        'EidAlAdha',
    'event_fathers_day_28':        "Father's day",
    'event_halloween_28':          'Halloween',
    'event_mothers_day_28':        "Mother's day",
    'event_newyear_28':            'NewYear',
    'event_orthodox_christmas_28': 'OrthodoxChristmas',
    'event_orthodox_easter_28':    'OrthodoxEaster',
    'event_ramadan_starts_28':     'Ramadan starts',
    'event_thanksgiving_28':       'Thanksgiving',
    'event_valentines_day_28':     'ValentinesDay',
    'event_superbowl_28':          'SuperBowl',
    'event_independence_day_28':   'IndependenceDay',
    'event_memorial_day_28':       'MemorialDay',
    'event_labor_day_28':          'LaborDay',
    'event_mlk_day_28':            'MartinLutherKingDay',
    'event_presidents_day_28':     'PresidentsDay',
    'event_columbus_day_28':       'ColumbusDay',
    'event_veterans_day_28':       'VeteransDay',
    'event_st_patricks_day_28':    'StPatricksDay',
    'event_cinco_de_mayo_28':      'Cinco De Mayo',
    'event_chanukah_28':           'Chanukah End',
    'event_lent_start_28':         'LentStart',
    'event_lent_week2_28':         'LentWeek2',
    'event_pesach_end_28':         'Pesach End',
    'event_purim_end_28':          'Purim End',
}

for col, name in EVENT_MAP.items():
    long_df[col] = (
        (long_df['event_name_1'] == name) | (long_df['event_name_2'] == name)
    ).astype(int)

long_df['is_month_end'] = long_df['date'].dt.is_month_end.astype(int)
print(f'Event columns built: {len(EVENT_MAP)}')

Event columns built: 28


## 8) Assign 28-Day Window Index per Item

In [8]:
long_df = long_df.sort_values(['item_id', 'date']).reset_index(drop=True)
long_df['window_idx'] = long_df.groupby('item_id').cumcount() // WINDOW
print(f'Windows per item: {long_df.groupby("item_id")["window_idx"].max().mean():.0f}')

Windows per item: 68


## 9) Aggregate to 28-Day Rows

In [9]:
event_cols = list(EVENT_MAP.keys())

agg_dict = {
    'date':                   ('date',         'first'),
    TARGET:                   ('sales',         'sum'),
    'aggregated_sell_price':  ('sell_price',    'mean'),
    'is_month_end':           ('is_month_end',  'max'),
    'snap_ca':                ('snap_CA',       'max'),
}
for col in event_cols:
    agg_dict[col] = (col, 'max')

agg = (
    long_df.groupby(['item_id', 'window_idx'])
    .agg(**agg_dict)
    .reset_index()
    .drop(columns='window_idx')
)

print(f'Aggregated shape : {agg.shape}')
print(f'Date range       : {agg["date"].min().date()} to {agg["date"].max().date()}')

Aggregated shape : (99153, 34)
Date range       : 2011-01-29 to 2016-04-16


## 10) Enforce Column Order

In [10]:
FINAL_COLS = [
    'item_id', 'date', TARGET,
    'is_month_end', 'aggregated_sell_price', 'snap_ca',
] + event_cols

agg = agg[FINAL_COLS].copy()
print(f'Final columns : {len(agg.columns)}')
print(f'Null values   : {agg.isna().sum().sum()}')

Final columns : 34
Null values   : 0


## 11) 70 / 15 / 15 Time-Based Split

In [11]:
dates = sorted(agg['date'].unique())
n     = len(dates)

train_dates = dates[:int(0.70 * n)]
val_dates   = dates[int(0.70 * n):int(0.85 * n)]
test_dates  = dates[int(0.85 * n):]

train_df = agg[agg['date'].isin(train_dates)].reset_index(drop=True)
val_df   = agg[agg['date'].isin(val_dates)].reset_index(drop=True)
test_df  = agg[agg['date'].isin(test_dates)].reset_index(drop=True)

print(f'Total unique dates : {n}')
print(f'Train : {len(train_dates)} windows | {len(train_df):,} rows | {pd.Timestamp(train_dates[0]).date()} to {pd.Timestamp(train_dates[-1]).date()}')
print(f'Val   : {len(val_dates)} windows | {len(val_df):,} rows | {pd.Timestamp(val_dates[0]).date()} to {pd.Timestamp(val_dates[-1]).date()}')
print(f'Test  : {len(test_dates)} windows | {len(test_df):,} rows | {pd.Timestamp(test_dates[0]).date()} to {pd.Timestamp(test_dates[-1]).date()}')

Total unique dates : 69
Train : 48 windows | 68,976 rows | 2011-01-29 to 2014-09-06
Val   : 10 windows | 14,370 rows | 2014-10-04 to 2015-06-13
Test  : 11 windows | 15,807 rows | 2015-07-11 to 2016-04-16


## 12) Sanity Checks

In [12]:
assert len(set(train_dates) & set(val_dates))  == 0, 'OVERLAP: train/val'
assert len(set(val_dates)   & set(test_dates)) == 0, 'OVERLAP: val/test'
assert set(train_df['item_id']) == set(agg['item_id']), 'Missing items in train'
assert agg['snap_ca'].isin([0, 1]).all(), 'snap_ca not binary'
assert agg.isna().sum().sum() == 0, 'Null values found'
print('All checks passed.')
print(f'Items : {agg["item_id"].nunique()}')
print(f'\n{TARGET} summary:')
print(agg[TARGET].describe().round(2))

All checks passed.
Items : 1437

aggregated_sales_28 summary:
count    99153.00
mean        54.27
std        126.56
min          0.00
25%          0.00
50%         22.00
75%         54.00
max       5348.00
Name: aggregated_sales_28, dtype: float64


## 13) Save to Disk

In [13]:
train_df.to_csv(OUT / 'train.csv', index=False)
val_df.to_csv(OUT   / 'val.csv',   index=False)
test_df.to_csv(OUT  / 'test.csv',  index=False)

print(f'Saved to {OUT}')
print(f'  train.csv : {len(train_df):,} rows')
print(f'  val.csv   : {len(val_df):,} rows')
print(f'  test.csv  : {len(test_df):,} rows')

Saved to C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment\data\processed\28_Day_Dataset
  train.csv : 68,976 rows
  val.csv   : 14,370 rows
  test.csv  : 15,807 rows
